# FedKG-Maint — Phase 3 Closure Audit v1

## الغرض

هذا **تدقيق نهائي لنتائج v3**؛ لا يعيد تدريب أي نموذج، ولا يختار إعدادات جديدة من الاختبار. سيقرأ ملفات التنبؤ المحفوظة، يعيد حساب PR-AUC، يتحقق من توافق الملفات والإعدادات، ينفّذ مقارنة paired bootstrap بين `DP-FedAvg-CompactLinear` و`ClippedFedAvg` مع إعادة أخذ العينات على مستوى الماكينة، ويعيد حساب ميزانية الخصوصية بواسطة `dp-accounting` إن أمكن.

**حد مهم:** مجموعة `test_site_full` استُخدمت في تجارب سابقة. لذلك هذا تدقيق على benchmark الحالي، وليس اختبارًا خارجيًا جديدًا مستقلًا، ولا يجوز استخدامه لاختيار إعدادات إضافية.

شغّل الخلايا بالترتيب باستخدام **Run all** في بيئة Colab التي تحتوي على مجلد Google Drive `FedKG-Maint` نفسه. المخرجات ستُحفظ في `phase3_closure_audit_v1/`.

## 1. إعداد المسارات والتحقق من الملفات

In [ ]:
import os, sys, json, math, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/FedKG-Maint")
except ImportError:
    PROJECT_DIR = Path(os.environ.get("FEDKG_ROOT", "./FedKG-Maint")).resolve()

PREP_DIR = PROJECT_DIR / "prep_v3"
P3_DIR = PROJECT_DIR / "phase3_federated_prep_v3_privacy_v2"
V3_DIR = PROJECT_DIR / "phase3_federated_privacy_fix_v3"
AUDIT_DIR = PROJECT_DIR / "phase3_closure_audit_v1"
for sub in ["results", "reports", "logs"]:
    (AUDIT_DIR / sub).mkdir(parents=True, exist_ok=True)

SEEDS = [42, 43, 44]
METHODS = ["DP-FedAvg-CompactLinear", "ClippedFedAvg", "FedAvg", "CentralCompactLinear"]
PRIMARY_DP = "DP-FedAvg-CompactLinear"
PRIMARY_CONTROL = "ClippedFedAvg"
ROUNDS = 3
DELTA = 1e-5
EPSILON_CAP = 10.0
BOOTSTRAP_B = 2000
BOOTSTRAP_RANDOM_SEED = 20261017

required = [PREP_DIR / "fedkg_data.py",
            V3_DIR / "reports" / "compact_dp_selection_lock.json",
            V3_DIR / "results" / "compact_dp_final_test_results.csv",
            V3_DIR / "results" / "compact_dp_final_summary.csv"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Required Phase 1/Phase 2/v3 outputs are missing:\n" + "\n".join(missing))

PRED_DIR = V3_DIR / "predictions"
missing_predictions = [str(PRED_DIR / f"{m}__seed{s}_test_site_full.parquet")
                      for m in METHODS for s in SEEDS
                      if not (PRED_DIR / f"{m}__seed{s}_test_site_full.parquet").exists()]
if missing_predictions:
    raise FileNotFoundError("Missing saved test predictions; rerun Phase 3 v3 before the audit:\n" + "\n".join(missing_predictions))

print("Project:", PROJECT_DIR)
print("v3 results:", V3_DIR)
print("Audit outputs:", AUDIT_DIR)
print("Found all", len(METHODS) * len(SEEDS), "prediction files.")

## 2. إعادة حساب ميزانية الخصوصية بمحاسب مستقل

نحسب epsilon بطريقتين: التحويل اليدوي التقليدي من RDP إلى (ε, δ)-DP، وحساب مكتبة `dp-accounting`. قد تختلف النتيجتان لأن طريقة التحويل العددي من RDP إلى DP ليست بالضرورة متطابقة. لذلك **لا نشترط تساوي الرقمين**؛ نشترط أن تكون نتيجة المكتبة متسقة مع كونها ليست أضعف من الحد اليدوي المستخدم هنا، وأن يبقى كل من الحد اليدوي ونتيجة المكتبة دون سقف الدراسة `ε ≤ 10`. نحتفظ بالرقم اليدوي الأكثر تحفظًا في تقرير التحقق، ونُظهر رقم المكتبة للمقارنة المستقلة.

هذا لا يغيّر آلية الخصوصية أو التدريب؛ إنه يصلح شرط التحقق فقط. تظل المحاسبة محدودة بقناة تحديثات العملاء وبالافتراضات المبيّنة أدناه.

In [ ]:
# Prefer Google's dp-accounting implementation. If absent, try to install a pinned version.
DP_ACCOUNTING_AVAILABLE = False
DP_ACCOUNTING_IMPORT_ERROR = None
try:
    from dp_accounting import dp_event, privacy_accountant
    from dp_accounting.rdp import rdp_privacy_accountant
    DP_ACCOUNTING_AVAILABLE = True
except Exception as e:
    DP_ACCOUNTING_IMPORT_ERROR = repr(e)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dp-accounting==0.6.0"],
                       check=True, timeout=180)
        from dp_accounting import dp_event, privacy_accountant
        from dp_accounting.rdp import rdp_privacy_accountant
        DP_ACCOUNTING_AVAILABLE = True
    except Exception as e2:
        DP_ACCOUNTING_IMPORT_ERROR = repr(e2)

# Match the order grid used in v3 first, then use a denser order grid for a tighter bound.
V3_ORDERS = np.array([1.25, 1.5, 2, 3, 4, 5, 8, 10, 16, 32, 64, 128, 256, 512], dtype=float)
DENSE_ORDERS = np.unique(np.concatenate([
    np.arange(1.01, 10.0, 0.01),
    np.arange(10.0, 100.0, 0.1),
    np.array([128.0, 256.0, 512.0, 1024.0])
]))

def manual_rdp_epsilon(noise_multiplier, steps, delta, orders):
    # Exact RDP of a Gaussian mechanism, basic composition, and standard RDP-to-(eps,delta)-DP conversion.
    rdp = steps * orders / (2.0 * noise_multiplier ** 2)
    vals = rdp + math.log(1.0 / delta) / (orders - 1.0)
    idx = int(np.argmin(vals))
    return float(vals[idx]), float(orders[idx])

def library_rdp_epsilon(noise_multiplier, steps, delta, orders):
    accountant = rdp_privacy_accountant.RdpAccountant(
        orders=list(map(float, orders)),
        neighboring_relation=privacy_accountant.NeighboringRelation.ADD_OR_REMOVE_ONE)
    accountant.compose(dp_event.SelfComposedDpEvent(
        dp_event.GaussianDpEvent(float(noise_multiplier)), int(steps)))
    return float(accountant.get_epsilon(float(delta)))

lock = json.loads((V3_DIR / "reports" / "compact_dp_selection_lock.json").read_text())
final_results = pd.read_csv(V3_DIR / "results" / "compact_dp_final_test_results.csv")
final_summary_saved = pd.read_csv(V3_DIR / "results" / "compact_dp_final_summary.csv")
selected_sigma = float(lock["selected_noise_multiplier"])
selected_clip = float(lock["selected_clip_norm"])
selected_local_epochs = int(lock["selected_local_epochs"])
lock_epsilon = float(lock["selected_epsilon"])

manual_v3_eps, manual_v3_order = manual_rdp_epsilon(selected_sigma, ROUNDS, DELTA, V3_ORDERS)
manual_dense_eps, manual_dense_order = manual_rdp_epsilon(selected_sigma, ROUNDS, DELTA, DENSE_ORDERS)
if DP_ACCOUNTING_AVAILABLE:
    library_v3_eps = library_rdp_epsilon(selected_sigma, ROUNDS, DELTA, V3_ORDERS)
    library_dense_eps = library_rdp_epsilon(selected_sigma, ROUNDS, DELTA, DENSE_ORDERS)
else:
    library_v3_eps = np.nan
    library_dense_eps = np.nan

privacy_audit = {
    "selected_noise_multiplier": selected_sigma,
    "selected_clip_norm": selected_clip,
    "selected_local_epochs": selected_local_epochs,
    "rounds": ROUNDS,
    "delta": DELTA,
    "epsilon_cap": EPSILON_CAP,
    "epsilon_stored_in_selection_lock": lock_epsilon,
    "epsilon_recomputed_v3_order_grid": manual_v3_eps,
    "optimal_order_v3_grid": manual_v3_order,
    "epsilon_recomputed_dense_order_grid": manual_dense_eps,
    "optimal_order_dense_grid": manual_dense_order,
    "dp_accounting_package_available": DP_ACCOUNTING_AVAILABLE,
    "dp_accounting_package_version": getattr(sys.modules.get("dp_accounting"), "__version__", "unknown") if DP_ACCOUNTING_AVAILABLE else None,
    "epsilon_dp_accounting_same_grid": library_v3_eps,
    "epsilon_dp_accounting_dense_grid": library_dense_eps,
    "accountant_import_or_install_error": DP_ACCOUNTING_IMPORT_ERROR,
    "privacy_scope": "client-update mechanism only; add/remove-one client adjacency; trusted server; fixed denominator; full participation; no subsampling amplification"
}
print(json.dumps(privacy_audit, indent=2))

assert np.isclose(lock_epsilon, manual_v3_eps, atol=1e-8), "Stored epsilon does not reproduce from the v3 order grid."
assert manual_dense_eps <= manual_v3_eps + 1e-8, "Dense-order RDP bound should not be looser than the sparse-grid bound."
assert manual_v3_eps <= EPSILON_CAP + 1e-8, "Privacy accountant exceeds the predeclared epsilon cap."
if DP_ACCOUNTING_AVAILABLE:
    # Do not require numerical equality: the library and the manual code use different
    # RDP-to-(epsilon, delta)-DP conversions. Check the direction of the bound and cap instead.
    assert np.isfinite(library_v3_eps) and library_v3_eps <= manual_v3_eps + 1e-7, (
        f"Library epsilon should be no looser than the manual bound: {library_v3_eps} > {manual_v3_eps}"
    )
    assert np.isfinite(library_dense_eps) and library_dense_eps <= manual_dense_eps + 1e-6, (
        f"Library dense-grid epsilon should be no looser than the manual bound: {library_dense_eps} > {manual_dense_eps}"
    )
    assert library_v3_eps <= EPSILON_CAP + 1e-7, "Library epsilon exceeds the predeclared epsilon cap."
    assert library_dense_eps <= EPSILON_CAP + 1e-6, "Dense-grid library epsilon exceeds the predeclared epsilon cap."
else:
    print("WARNING: Google dp-accounting could not be imported/installed. Manual RDP recomputation completed, but library cross-check is unavailable.")

## 3. مطابقة التنبؤات المحفوظة مع بيانات الاختبار والنتائج المسجلة

In [ ]:
# Import the project data loader so the label and row order are taken from the same saved split.
sys.path.insert(0, str(PREP_DIR))
import importlib
sys.modules.pop("fedkg_data", None)
importlib.invalidate_caches()
import fedkg_data

Xtest_all, ytest, mtest = fedkg_data.load_split("test_site_full", policy="global", feature_set="full")
ytest = np.asarray(ytest).reshape(-1).astype(np.int8)
meta = mtest.reset_index(drop=True).copy()
assert "machineID" in meta.columns and "datetime" in meta.columns
if "client_id" in meta.columns:
    assert set(meta["client_id"].astype(str).unique()) == {"factory_5"}, "Test rows are not exclusively factory_5."

meta_machines = meta["machineID"].astype(str).to_numpy()
meta_dates = pd.to_datetime(meta["datetime"], utc=True).astype("int64").to_numpy()
assert len(ytest) == len(meta) == len(Xtest_all)
assert np.isfinite(ytest).all()

PRED = {}
computed_rows = []
consistency_rows = []
for method in METHODS:
    for seed in SEEDS:
        path = PRED_DIR / f"{method}__seed{seed}_test_site_full.parquet"
        pred = pd.read_parquet(path)
        required_cols = {"machineID", "datetime", "score", "seed"}
        assert required_cols.issubset(pred.columns), f"Missing required prediction columns in {path.name}"
        assert len(pred) == len(meta), f"Wrong prediction row count in {path.name}"
        assert pred["seed"].nunique() == 1 and int(pred["seed"].iloc[0]) == seed
        assert np.array_equal(pred["machineID"].astype(str).to_numpy(), meta_machines), f"Machine row order mismatch: {path.name}"
        pred_dates = pd.to_datetime(pred["datetime"], utc=True).astype("int64").to_numpy()
        assert np.array_equal(pred_dates, meta_dates), f"Timestamp row order mismatch: {path.name}"
        scores = pred["score"].to_numpy(dtype=np.float64)
        assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
        ap = float(average_precision_score(ytest, scores))
        saved = final_results[(final_results["method"] == method) & (final_results["seed"] == seed)]
        assert len(saved) == 1, f"Expected exactly one saved result row for {method} seed {seed}"
        saved_ap = float(saved["test_site_full_AP"].iloc[0])
        delta_ap = ap - saved_ap
        consistency_rows.append({"method": method, "seed": seed, "recomputed_AP_from_saved_predictions": ap,
                                 "AP_in_original_results_csv": saved_ap, "absolute_difference": abs(delta_ap)})
        computed_rows.append({"method": method, "seed": seed, "recomputed_test_AP": ap,
                              "test_prevalence": float(ytest.mean()), "n_test_rows": len(ytest),
                              "n_machines": int(meta["machineID"].nunique())})
        PRED[(method, seed)] = {"scores": scores, "path": str(path)}

AP_CHECK = pd.DataFrame(consistency_rows)
AP_TABLE = pd.DataFrame(computed_rows)
AP_CHECK.to_csv(AUDIT_DIR / "results" / "prediction_ap_reproducibility.csv", index=False)
AP_TABLE.to_csv(AUDIT_DIR / "results" / "recomputed_test_ap_by_seed.csv", index=False)

max_ap_reproduction_error = float(AP_CHECK["absolute_difference"].max())
SUMMARY_RECALC = (AP_TABLE.groupby("method", as_index=False)
                  .agg(seeds=("seed", "nunique"), mean_test_AP=("recomputed_test_AP", "mean"),
                       sd_test_AP=("recomputed_test_AP", "std")))
SUMMARY_CHECK = SUMMARY_RECALC.merge(final_summary_saved[["method", "seeds", "mean_test_AP", "sd_test_AP"]],
                                     on="method", suffixes=("_recomputed", "_saved"), validate="one_to_one")
SUMMARY_CHECK["mean_AP_abs_error"] = (SUMMARY_CHECK["mean_test_AP_recomputed"] - SUMMARY_CHECK["mean_test_AP_saved"]).abs()
SUMMARY_CHECK["sd_AP_abs_error"] = (SUMMARY_CHECK["sd_test_AP_recomputed"] - SUMMARY_CHECK["sd_test_AP_saved"]).abs()
max_summary_error = float(SUMMARY_CHECK[["mean_AP_abs_error", "sd_AP_abs_error"]].to_numpy().max())
SUMMARY_CHECK.to_csv(AUDIT_DIR / "results" / "summary_ap_reproducibility.csv", index=False)

print("Test rows:", len(ytest), "| machines:", meta["machineID"].nunique(), "| prevalence:", round(float(ytest.mean()), 6))
print("Held-out factory:", sorted(meta["client_id"].astype(str).unique()) if "client_id" in meta.columns else "client_id absent")
print("Largest saved-prediction AP reproduction difference:", max_ap_reproduction_error)
print("Largest saved-summary AP/SD reproduction difference:", max_summary_error)
print("\nRecomputed AP by method and seed:")
print(AP_TABLE.pivot(index="seed", columns="method", values="recomputed_test_AP").to_string(float_format=lambda x:f"{x:.4f}"))
print("\nSummary AP cross-check:")
print(SUMMARY_CHECK.to_string(index=False, float_format=lambda x:f"{x:.6g}"))

# AP should reproduce exactly when the saved prediction scores retain their original precision.
assert max_ap_reproduction_error <= 1e-6, "Recomputed AP from saved scores does not match the recorded metrics; investigate score/data alignment."
assert max_summary_error <= 1e-6, "Recomputed AP mean/SD does not match the saved summary CSV."

## 4. Paired bootstrap على مستوى الماكينة

In [ ]:
# Primary comparison is DP vs clipped-noise-free FL. Same rows, same machines, same seeds.
# Cluster bootstrap resamples machines with replacement so within-machine time rows stay together.
cluster_codes, cluster_ids = pd.factorize(meta["machineID"].astype(str), sort=True)
n_clusters = len(cluster_ids)
assert n_clusters == 20, f"Expected 20 held-out machines; found {n_clusters}."


def make_ap_cache(y, scores, cluster_codes):
    order = np.argsort(-scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_y = y[order].astype(np.float64)
    sorted_clusters = cluster_codes[order]
    new_group = np.r_[True, sorted_scores[1:] != sorted_scores[:-1]]
    group_codes = np.cumsum(new_group, dtype=np.int64) - 1
    return {"groups": group_codes, "y": sorted_y, "clusters": sorted_clusters,
            "n_groups": int(group_codes[-1]) + 1}

def weighted_ap_from_cache(cache, cluster_weights):
    row_weights = cluster_weights[cache["clusters"]].astype(np.float64)
    group_total = np.bincount(cache["groups"], weights=row_weights, minlength=cache["n_groups"])
    group_positive = np.bincount(cache["groups"], weights=row_weights * cache["y"], minlength=cache["n_groups"])
    total_positive = float(group_positive.sum())
    if total_positive <= 0:
        return np.nan
    cum_total = np.cumsum(group_total)
    cum_positive = np.cumsum(group_positive)
    precision_at_group = cum_positive / np.maximum(cum_total, 1e-300)
    return float(np.sum((group_positive / total_positive) * precision_at_group))

AP_CACHE = {}
for method in METHODS:
    for seed in SEEDS:
        score = PRED[(method, seed)]["scores"]
        cache = make_ap_cache(ytest, score, cluster_codes)
        check_ap = weighted_ap_from_cache(cache, np.ones(n_clusters, dtype=np.float64))
        sklearn_ap = float(average_precision_score(ytest, score))
        if not np.isclose(check_ap, sklearn_ap, atol=1e-10, rtol=1e-10):
            raise AssertionError(f"Weighted-AP implementation check failed for {method}, seed {seed}: {check_ap} vs {sklearn_ap}")
        AP_CACHE[(method, seed)] = cache

rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)
seed_diffs = np.full((BOOTSTRAP_B, len(SEEDS)), np.nan, dtype=np.float64)
mean_diffs = np.full(BOOTSTRAP_B, np.nan, dtype=np.float64)
mean_dp_boot = np.full(BOOTSTRAP_B, np.nan, dtype=np.float64)
mean_ctrl_boot = np.full(BOOTSTRAP_B, np.nan, dtype=np.float64)
for b in range(BOOTSTRAP_B):
    sampled_clusters = rng.integers(0, n_clusters, size=n_clusters)
    cluster_weights = np.bincount(sampled_clusters, minlength=n_clusters).astype(np.float64)
    dp_values=[]; ctrl_values=[]
    for j, seed in enumerate(SEEDS):
        dp_ap = weighted_ap_from_cache(AP_CACHE[(PRIMARY_DP, seed)], cluster_weights)
        ctrl_ap = weighted_ap_from_cache(AP_CACHE[(PRIMARY_CONTROL, seed)], cluster_weights)
        seed_diffs[b, j] = dp_ap - ctrl_ap
        dp_values.append(dp_ap); ctrl_values.append(ctrl_ap)
    mean_dp_boot[b] = float(np.mean(dp_values))
    mean_ctrl_boot[b] = float(np.mean(ctrl_values))
    mean_diffs[b] = float(np.mean(seed_diffs[b, :]))

finite = np.isfinite(mean_diffs)
boot_diff = mean_diffs[finite]
boot_ci = np.quantile(boot_diff, [0.025, 0.975])
point_dp_aps = np.array([float(AP_TABLE[(AP_TABLE.method == PRIMARY_DP) & (AP_TABLE.seed == s)]["recomputed_test_AP"].iloc[0]) for s in SEEDS])
point_ctrl_aps = np.array([float(AP_TABLE[(AP_TABLE.method == PRIMARY_CONTROL) & (AP_TABLE.seed == s)]["recomputed_test_AP"].iloc[0]) for s in SEEDS])
point_diff_each_seed = point_dp_aps - point_ctrl_aps
point_diff_mean = float(np.mean(point_diff_each_seed))

paired_rows=[]
for j, seed in enumerate(SEEDS):
    ci_seed = np.quantile(seed_diffs[np.isfinite(seed_diffs[:, j]), j], [0.025, 0.975])
    paired_rows.append({"comparison": f"{PRIMARY_DP} minus {PRIMARY_CONTROL}", "seed": seed,
                        "DP_AP": point_dp_aps[j], "control_AP": point_ctrl_aps[j],
                        "observed_AP_difference": point_diff_each_seed[j],
                        "paired_machine_bootstrap_CI_low": ci_seed[0],
                        "paired_machine_bootstrap_CI_high": ci_seed[1]})
paired_rows.append({"comparison": f"{PRIMARY_DP} minus {PRIMARY_CONTROL}", "seed": "mean_over_seeds",
                    "DP_AP": float(point_dp_aps.mean()), "control_AP": float(point_ctrl_aps.mean()),
                    "observed_AP_difference": point_diff_mean,
                    "paired_machine_bootstrap_CI_low": boot_ci[0],
                    "paired_machine_bootstrap_CI_high": boot_ci[1]})
PAIRED = pd.DataFrame(paired_rows)
PAIRED["bootstrap_replicates"] = BOOTSTRAP_B
PAIRED["bootstrap_unit"] = "machineID (20 held-out machines), paired across methods and seeds"
PAIRED.to_csv(AUDIT_DIR / "results" / "paired_machine_bootstrap_dp_vs_clipped.csv", index=False)

print("Primary paired comparison: DP - ClippedFedAvg")
print(PAIRED.to_string(index=False, float_format=lambda x:f"{x:.4f}"))
print("Bootstrap probability that mean difference <= 0:", round(float(np.mean(boot_diff <= 0)), 4),
      "(descriptive bootstrap tail fraction, not a formal p-value)")
print("95% machine-cluster paired bootstrap CI for mean AP difference:", (float(boot_ci[0]), float(boot_ci[1])))

# Descriptive, predeclared screens from v3 — report them, do not use bootstrap result to tune the model.
prevalence = float(np.mean(ytest))
dp_mean = float(point_dp_aps.mean())
dp_sd = float(point_dp_aps.std(ddof=1))
utility_floor = 3.0 * prevalence
utility_screen_pass = bool(dp_mean >= utility_floor)
stability_screen_pass = bool(np.all(point_dp_aps > prevalence) and dp_sd <= max(dp_mean, 1e-12))
print("\nV3 mean-utility screen:", utility_screen_pass, "| mean AP:", round(dp_mean,4), "| 3x prevalence:", round(utility_floor,4))
print("V3 descriptive seed-stability screen:", stability_screen_pass, "| seed SD:", round(dp_sd,4), "| mean AP:", round(dp_mean,4))
print("DP-minus-clipped CI excludes zero:", bool(boot_ci[0] > 0 or boot_ci[1] < 0))

## 5. إصدار تقرير التدقيق النهائي

In [ ]:
checks = {}
def check(name, ok, detail=""):
    checks[name] = {"status": "PASS" if bool(ok) else "FAIL", "detail": str(detail)}
    print(f"[{'PASS' if ok else 'FAIL'}] {name}: {detail}")

check("AUDIT-1 v3 selection lock exists and is readable", (V3_DIR / "reports" / "compact_dp_selection_lock.json").exists())
check("AUDIT-2 all 12 prediction files are aligned to the held-out test rows",
      len(PRED) == 12 and set(meta["client_id"].astype(str).unique()) == {"factory_5"},
      f"predictions={len(PRED)}; held_out={sorted(meta['client_id'].astype(str).unique()) if 'client_id' in meta.columns else 'unknown'}")
check("AUDIT-3 reconstructed AP matches saved Phase 3 v3 metrics",
      max_ap_reproduction_error <= 1e-6, f"max_abs_error={max_ap_reproduction_error:.12g}")
check("AUDIT-4 recomputed summary mean/SD matches saved summary CSV",
      max_summary_error <= 1e-6, f"max_abs_error={max_summary_error:.12g}")
check("AUDIT-5 stored privacy epsilon reproduces", np.isclose(lock_epsilon, manual_v3_eps, atol=1e-8),
      f"stored={lock_epsilon:.6f}; recalculated={manual_v3_eps:.6f}")
check("AUDIT-6 dense-order RDP bound meets epsilon cap", manual_dense_eps <= EPSILON_CAP,
      f"epsilon_dense={manual_dense_eps:.6f}; cap={EPSILON_CAP}")
check("AUDIT-7 independent dp-accounting library cross-check available and consistent",
      DP_ACCOUNTING_AVAILABLE
      and np.isfinite(library_v3_eps) and library_v3_eps <= manual_v3_eps + 1e-7
      and np.isfinite(library_dense_eps) and library_dense_eps <= manual_dense_eps + 1e-6
      and library_v3_eps <= EPSILON_CAP + 1e-7
      and library_dense_eps <= EPSILON_CAP + 1e-6,
      f"available={DP_ACCOUNTING_AVAILABLE}; manual_v3={manual_v3_eps:.6f}; library_v3={library_v3_eps:.6f}; "
      f"manual_dense={manual_dense_eps:.6f}; library_dense={library_dense_eps:.6f}; cap={EPSILON_CAP}")
check("AUDIT-8 v3 mean-utility screen", utility_screen_pass,
      f"mean_DP_AP={dp_mean:.4f}; required_reference={utility_floor:.4f}")
check("AUDIT-9 v3 descriptive stability screen", stability_screen_pass,
      f"DP seed APs={point_dp_aps.tolist()}; sample_sd={dp_sd:.4f}")
check("AUDIT-10 paired machine bootstrap completed", len(PAIRED) == 4,
      f"B={BOOTSTRAP_B}; n_machines={n_clusters}; mean_diff={point_diff_mean:.4f}; CI=({boot_ci[0]:.4f},{boot_ci[1]:.4f})")
check("AUDIT-11 v3 selection lock declares validation-only selection",
      lock.get("selection_split") == "validation only" and lock.get("test_loaded_before_selection") is False,
      "This checks the saved protocol declaration; it is not a cryptographic proof of execution order.")

all_critical_pass = all(checks[k]["status"] == "PASS" for k in [
    "AUDIT-1 v3 selection lock exists and is readable",
    "AUDIT-2 all 12 prediction files are aligned to the held-out test rows",
    "AUDIT-3 reconstructed AP matches saved Phase 3 v3 metrics",
    "AUDIT-4 recomputed summary mean/SD matches saved summary CSV",
    "AUDIT-5 stored privacy epsilon reproduces",
    "AUDIT-6 dense-order RDP bound meets epsilon cap",
    "AUDIT-7 independent dp-accounting library cross-check available and consistent",
    "AUDIT-8 v3 mean-utility screen",
    "AUDIT-9 v3 descriptive stability screen",
    "AUDIT-10 paired machine bootstrap completed",
    "AUDIT-11 v3 selection lock declares validation-only selection"
])
ci_excludes_zero = bool(boot_ci[0] > 0 or boot_ci[1] < 0)
if all_critical_pass and boot_ci[0] > 0:
    conclusion = "The v3 utility screens and integrity/accounting checks pass; the paired machine-bootstrap CI is above zero, showing a benchmark-specific DP-vs-clipped AP advantage. This is not proof of end-to-end DP or external generalization."
elif all_critical_pass and boot_ci[1] < 0:
    conclusion = "The v3 utility screens and integrity/accounting checks pass, but the paired machine-bootstrap CI favors ClippedFedAvg. Do not claim DP utility parity based on this audit alone."
elif all_critical_pass:
    conclusion = "The v3 utility screens and integrity/accounting checks pass, but the paired machine-bootstrap CI includes zero. The observed positive mean difference is inconclusive; do not claim statistically clear superiority of DP over ClippedFedAvg."
else:
    conclusion = "The audit did not fully pass. Resolve every FAIL above before declaring the Phase 3 DP utility audit closed."

report = {
    "title": "FedKG-Maint Phase 3 Closure Audit v1",
    "source_run": "Phase 3 Privacy Utility Fix v3",
    "test_set": "test_site_full / held-out factory_5",
    "warning": "This test set was used in earlier experiments. Audit is not an untouched external replication.",
    "n_test_rows": int(len(ytest)), "n_machines": int(n_clusters),
    "test_prevalence": prevalence, "selected_clip_norm": selected_clip,
    "selected_noise_multiplier": selected_sigma, "selected_local_epochs": selected_local_epochs,
    "rounds": ROUNDS, "delta": DELTA, "epsilon_cap": EPSILON_CAP,
    "epsilon_stored_in_lock": lock_epsilon, "epsilon_v3_grid_recomputed": manual_v3_eps,
    "epsilon_dense_grid_recomputed": manual_dense_eps,
    "epsilon_dp_accounting_v3_grid": library_v3_eps,
    "epsilon_dp_accounting_dense_grid": library_dense_eps,
    "dp_accounting_library_available": DP_ACCOUNTING_AVAILABLE,
    "accountant_crosscheck_rule": "Do not require numeric equality across RDP-to-DP conversion implementations; require library epsilon to be no greater than the corresponding manual upper bound (within tolerance) and both to satisfy epsilon cap.",
    "per_seed_dp_AP": {str(s): float(point_dp_aps[i]) for i, s in enumerate(SEEDS)},
    "per_seed_clipped_AP": {str(s): float(point_ctrl_aps[i]) for i, s in enumerate(SEEDS)},
    "mean_dp_AP": dp_mean, "sd_dp_AP_sample": dp_sd,
    "mean_clipped_AP": float(point_ctrl_aps.mean()),
    "mean_paired_AP_difference_DP_minus_clipped": point_diff_mean,
    "paired_machine_bootstrap_ci_95_mean_difference": [float(boot_ci[0]), float(boot_ci[1])],
    "bootstrap_probability_difference_le_zero_descriptive": float(np.mean(boot_diff <= 0)),
    "bootstrap_replicates": BOOTSTRAP_B,
    "mean_utility_screen_pass": utility_screen_pass,
    "seed_stability_screen_pass": stability_screen_pass,
    "paired_ci_excludes_zero": ci_excludes_zero,
    "all_critical_checks_pass": all_critical_pass,
    "checks": checks,
    "conclusion": conclusion,
    "privacy_scope": "client-update channel only, add/remove-one client adjacency, trusted server, fixed denominator, full participation, no subsampling amplification; validation selection and preprocessing are outside the bound",
    "non_goals": ["No new model selection or hyperparameter tuning", "No retraining", "No claim of end-to-end DP", "No claim that SecureAggSim is cryptographic secure aggregation", "No claim of untouched external generalization"]
}
(AUDIT_DIR / "reports" / "phase3_closure_audit_report.json").write_text(json.dumps(report, indent=2, default=str))

md = [
    "# FedKG-Maint — Phase 3 Closure Audit v1", "",
    "> **Scope:** post-run audit of Phase 3 v3. It does not retrain or tune models. The held-out factory test was used previously, so the results are benchmark-specific, not a new external replication.", "",
    "## Selected configuration", "",
    f"- Local epochs: `{selected_local_epochs}` per round",
    f"- Clipping norm: `{selected_clip}`",
    f"- Noise multiplier: `{selected_sigma}`",
    f"- Rounds: `{ROUNDS}`",
    f"- Scoped epsilon: `{manual_v3_eps:.6f}` on v3 orders; `{manual_dense_eps:.6f}` on a dense RDP order grid at delta `{DELTA}`",
    f"- Independent `dp-accounting` RDP conversion: `{library_v3_eps:.6f}` on v3 orders; `{library_dense_eps:.6f}` on dense orders" if DP_ACCOUNTING_AVAILABLE else "- Independent `dp-accounting` cross-check unavailable; manual bound reported.",
    "- The manual and library conversions need not be numerically identical; audit checks that the library values are no looser than the manual bounds and both meet the epsilon cap.", "",
    "## Primary paired comparison", "",
    f"- DP mean test PR-AUC: `{dp_mean:.4f}` (sample SD `{dp_sd:.4f}`)",
    f"- ClippedFedAvg mean test PR-AUC: `{float(point_ctrl_aps.mean()):.4f}`",
    f"- Mean paired difference (DP - clipped): `{point_diff_mean:.4f}`",
    f"- 95% paired machine-cluster bootstrap CI: `[{boot_ci[0]:.4f}, {boot_ci[1]:.4f}]`",
    f"- Descriptive bootstrap fraction with difference <= 0: `{float(np.mean(boot_diff <= 0)):.4f}` (not a formal p-value)",
    "- Seeds compared: `42, 43, 44`; held-out machine clusters: `20`; bootstrap replicates: `2000`.", "",
    "## Audit checks", ""
]
for name, item in checks.items():
    md.append(f"- **{item['status']}** — {name}: {item['detail']}")
md += ["", "## Decision", "", conclusion, "",
       "## Interpretation limits", "",
       "- The epsilon claim covers the clipped/noised client-update channel only, under the stated add/remove adjacency and trusted-server assumptions.",
       "- Validation-based model selection, preprocessing statistics, evaluator access to raw data, and external diagnostic release are outside the stated epsilon claim.",
       "- `SecureAggSim` remains an arithmetic simulation, not cryptographic secure aggregation.",
       "- Three seeds and 20 held-out machines do not establish broad generalization. The bootstrap quantifies uncertainty over sampled held-out machines conditional on these trained models.",
       "- A paired CI including zero means the results do not establish clear superiority over the clipped baseline; it does not erase the observed mean utility result.", ""]
(AUDIT_DIR / "reports" / "phase3_closure_audit_report.md").write_text("\n".join(md))

pd.DataFrame([{"check": k, **v} for k, v in checks.items()]).to_csv(AUDIT_DIR / "results" / "phase3_closure_audit_checks.csv", index=False)
print("\n" + conclusion)
print("\nAudit reports saved to:", AUDIT_DIR / "reports")
print("Primary paired comparison saved to:", AUDIT_DIR / "results" / "paired_machine_bootstrap_dp_vs_clipped.csv")
if not all_critical_pass:
    print("\nAUDIT STATUS: NOT CLOSED — at least one required check failed. Review the FAIL rows.")
elif boot_ci[0] > 0:
    print("\nAUDIT STATUS: integrity/accounting + utility screens PASS; paired benchmark difference favors DP, subject to stated limitations.")
else:
    print("\nAUDIT STATUS: integrity/accounting + utility screens PASS; no statistically clear DP advantage over clipped baseline was established by this paired audit.")

## ماذا يعني الناتج؟

- لو كل فحوصات الملفات والميزانية والمنفعة اجتازت الاختبارات، فهذا يؤكد سلامة إعادة إنتاج نتائج v3 ضمن نطاقها.
- لو فاصل الثقة للفرق `DP - ClippedFedAvg` كله فوق الصفر، فهناك دليل benchmark-specific على فرق إيجابي في هذه المجموعة.
- لو فاصل الثقة يشمل الصفر، لا ندّعي تفوق DP إحصائيًا. نعرض متوسط التحسن كما هو، مع عدم اليقين.
- حتى عند النجاح، لا يعني ذلك اكتمال Secure Aggregation التشفيري أو وجود خصوصية شاملة end-to-end أو إثبات تعميم على مصانع جديدة.

المخرجات الرئيسية:
- `results/paired_machine_bootstrap_dp_vs_clipped.csv`
- `results/prediction_ap_reproducibility.csv`
- `results/summary_ap_reproducibility.csv`
- `results/phase3_closure_audit_checks.csv`
- `reports/phase3_closure_audit_report.md`
- `reports/phase3_closure_audit_report.json`